# HSAs → Daily Climate (CHIRPS + ERA5)

Exports one row per HSA per calendar day for the study period.
Output: `{OUT_DIR}/DRIVE_CLIMATE_BY_HSA_DOWNLOAD_DAILY/INF_HSA_<NAME>_daily.csv`

Columns: `FacilityName, date, P_precip, T_mean_C, T_max_C, T_min_C,
Td_C, DTR_C, wind_speed_ms, SM1, SM2, hours_above_30C, heat_index_C`

**Run once** before `Generate_Daily_Modeling_Dataset.ipynb`.


In [1]:
# ── BOUNDARY VERSION SELECTOR ──────────────────────────────────────────────
# Choose which HSA boundary bundle to use for this analysis.
# Must match a bundle produced by HSA_FINAL.ipynb.
#   'v6'  — original greedy algorithm (no post-selection corrections)
#   'v7'  — + anchor upgrade/demotion + major-orphan promotion
#   'v8'  — + satellite bubble boundaries
BOUNDARY_VERSION = "v8"   # change as needed
# ────────────────────────────────────────────────────────────────────────────


In [2]:
# STEP 0 — Earth Engine init + config
import ee, re, time, os
from datetime import date, timedelta

# CHANGE THIS to your own GEE project ID (see SETUP_INSTRUCTIONS.md)
PROJECT = "ee-izaslavsky"  # <-- replace with your project ID
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

# ── Date range ───────────────────────────────────────────────────────────────
# Start 14 days before first valid outcome date so lag windows are complete.
DAY_START = "2022-06-01"   # 14-day buffer before 2022-06-15 model start
DAY_END   = "2024-01-31"

NETWORK   = "INF"
MODE      = "footprint"
DATA_DIR  = os.environ.get("HSA_DATA_DIR", "data")
OUT_DIR   = os.environ.get("HSA_OUT_DIR", os.environ.get("PIPELINE_OUT_DIR", "out"))

TEST_MODE       = False   # full run: export every HSA and every day
TEST_HSA_COUNT  = 2
TEST_DAY_COUNT  = 5

COLL = {
    "CHIRPS":  "UCSB-CHG/CHIRPS/DAILY",
    "ERA5":    "ECMWF/ERA5_LAND/HOURLY",
}
SCALE = {"CHIRPS": 5550, "ERA5": 9000}

print(f"EE initialized  |  {DAY_START} → {DAY_END}")
print(f"BOUNDARY_VERSION = {BOUNDARY_VERSION}")
print(f"TEST_MODE = {TEST_MODE}")


EE initialized  |  2022-06-01 → 2024-01-31
BOUNDARY_VERSION = v8
TEST_MODE = False


In [3]:
# STEP 1 — Load HSA GeoJSON
import geopandas as gpd, geemap, os

GEOJSON_PATH = os.path.join(OUT_DIR, f"{NETWORK}_{MODE}_hsas_{BOUNDARY_VERSION}.geojson")
gdf = gpd.read_file(GEOJSON_PATH).to_crs(4326)
id_col = "FacilityName"
gdf = gdf[~gdf.geometry.is_empty][[id_col, "geometry"]].copy()

ee_fc_hsa = geemap.gdf_to_ee(gdf, geodesic=False)
ids = gdf[id_col].tolist()
print(f"Loaded {len(ids)} HSAs from {GEOJSON_PATH}")


Loaded 19 HSAs from out/INF_footprint_hsas_v8.geojson


In [4]:
# STEP 2 — Build daily date list
from datetime import date, timedelta

start = date.fromisoformat(DAY_START)
end   = date.fromisoformat(DAY_END)

all_days = []
d = start
while d <= end:
    all_days.append(d.isoformat())
    d += timedelta(days=1)

print(f"{len(all_days)} days: {all_days[0]} → {all_days[-1]}")


610 days: 2022-06-01 → 2024-01-31


In [5]:
# STEP 3 — Geometry helpers (reused from weekly notebook)
SIMPLIFY_M   = 10
CLEAN_BUF_M  = 1
MAX_ERROR_M  = 10
FALLBACK_M   = 250

def _with_retry(fn, label, retries=4, backoff=5):
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(backoff * (2 ** attempt))

def _force_planar(g):
    t = ee.String(g.type())
    c = g.coordinates()
    return ee.Geometry(ee.Algorithms.If(
        t.equals("Polygon"),
        ee.Geometry.Polygon(c, None, False),
        ee.Algorithms.If(
            t.equals("MultiPolygon"),
            ee.Geometry.MultiPolygon(c, None, False),
            g,
        ),
    ))

def robust_ee_geom(hid):
    feat = ee_fc_hsa.filter(ee.Filter.eq(id_col, hid)).first()
    g = _force_planar(ee.Feature(feat).geometry())
    g = g.buffer(0, MAX_ERROR_M).simplify(SIMPLIFY_M)
    if CLEAN_BUF_M:
        g = g.buffer(CLEAN_BUF_M, MAX_ERROR_M).buffer(-CLEAN_BUF_M, MAX_ERROR_M)
    area_ok = ee.Number(g.area(MAX_ERROR_M)).gt(0)
    fallback = g.centroid(MAX_ERROR_M).buffer(FALLBACK_M).bounds()
    g2 = ee.Geometry(ee.Algorithms.If(area_ok, g, fallback))
    _with_retry(lambda: g2.type().getInfo(), "geom")
    return g2

print("Geometry helpers ready")


Geometry helpers ready


In [6]:
# STEP 4 — Daily climate extraction function

def make_daily_fc(hsa_id, geom, days_list):
    """
    Returns ee.FeatureCollection: one Feature per day with all climate vars.
    Uses ERA5 scale (9000m) for all bands — CHIRPS is spatially averaged at
    that scale, which is fine for HSA-level analysis.
    """
    days_ee = ee.List(days_list)

    def process_day(date_str):
        d0 = ee.Date(date_str)
        d1 = d0.advance(1, "day")

        # CHIRPS precipitation (daily total)
        chirps = (ee.ImageCollection(COLL["CHIRPS"])
                  .filterDate(d0, d1)
                  .select("precipitation")
                  .map(lambda im: im.unmask(0))
                  .mean()
                  .rename("P_precip"))

        # ERA5 hourly → daily aggregates
        era5 = ee.ImageCollection(COLL["ERA5"]).filterDate(d0, d1)
        t    = era5.select("temperature_2m")
        td   = era5.select("dewpoint_temperature_2m")
        u    = era5.select("u_component_of_wind_10m")
        v    = era5.select("v_component_of_wind_10m")
        sm1  = era5.select("volumetric_soil_water_layer_1")
        sm2  = era5.select("volumetric_soil_water_layer_2")

        t_mean = t.mean().subtract(273.15).rename("T_mean_C")
        t_max  = t.max() .subtract(273.15).rename("T_max_C")
        t_min  = t.min() .subtract(273.15).rename("T_min_C")
        td_c   = td.mean().subtract(273.15).rename("Td_C")
        # DTR: max-min in Kelvin = max-min in Celsius
        dtr    = t.max().subtract(t.min()).rename("DTR_C")
        wspd   = u.mean().hypot(v.mean()).rename("wind_speed_ms")
        sm1_m  = sm1.mean().rename("SM1")
        sm2_m  = sm2.mean().rename("SM2")
        # Hours above 30°C (subtract 273.15 from each hourly image, then threshold)
        t_c    = t.map(lambda img: img.subtract(273.15))
        h30    = t_c.map(lambda img: img.gt(30)).sum().rename("hours_above_30C")
        # Simplified heat index: T + 0.4*(Td - T)  [all in Celsius]
        hi     = t_mean.add(td_c.subtract(t_mean).multiply(0.4)).rename("heat_index_C")

        img = (chirps
               .addBands(t_mean).addBands(t_max).addBands(t_min)
               .addBands(td_c).addBands(dtr).addBands(wspd)
               .addBands(sm1_m).addBands(sm2_m)
               .addBands(h30).addBands(hi))

        r = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=SCALE["ERA5"],
            maxPixels=1e9,
            bestEffort=True,
            tileScale=4,
        )

        return ee.Feature(None, {
            "FacilityName":   hsa_id,
            "date":           date_str,
            "P_precip":       r.get("P_precip"),
            "T_mean_C":       r.get("T_mean_C"),
            "T_max_C":        r.get("T_max_C"),
            "T_min_C":        r.get("T_min_C"),
            "Td_C":           r.get("Td_C"),
            "DTR_C":          r.get("DTR_C"),
            "wind_speed_ms":  r.get("wind_speed_ms"),
            "SM1":            r.get("SM1"),
            "SM2":            r.get("SM2"),
            "hours_above_30C": r.get("hours_above_30C"),
            "heat_index_C":   r.get("heat_index_C"),
        })

    return ee.FeatureCollection(days_ee.map(process_day))

print("Daily climate function ready")


Daily climate function ready


In [7]:
# STEP 5 — Export loop (one task per HSA)
import os

DRIVE_FOLDER = f"{NETWORK}_DAILY_CLIMATE_{BOUNDARY_VERSION.upper()}"

def _safe_name(s, maxlen=100):
    return re.sub(r"[^a-zA-Z0-9_\-]", "_", str(s))[:maxlen]

# Export every HSA in the current INF footprint GeoJSON.
# Keep this as the full ids list unless deliberately debugging one HSA.
ids_use  = ids
days_use = all_days

print(f"Exporting {len(ids_use)} HSAs × {len(days_use)} days → Drive/{DRIVE_FOLDER}")
print()

all_tasks = []
all_expected = []

for hsa_id in ids_use:
    time.sleep(3)
    try:
        geom = robust_ee_geom(hsa_id)
        fc   = make_daily_fc(hsa_id, geom, days_use)

        safe  = _safe_name(hsa_id)
        desc  = f"{NETWORK}_daily_{safe}"
        fname = f"{NETWORK}_HSA_{safe}_daily"

        task = ee.batch.Export.table.toDrive(
            collection=fc,
            description=desc[:100],
            folder=DRIVE_FOLDER,
            fileNamePrefix=fname,
            fileFormat="CSV",
            selectors=["FacilityName", "date",
                       "P_precip", "T_mean_C", "T_max_C", "T_min_C",
                       "Td_C", "DTR_C", "wind_speed_ms", "SM1", "SM2",
                       "hours_above_30C", "heat_index_C"],
        )
        task.start()
        all_tasks.append(task)
        all_expected.append(fname + ".csv")
        print(f"  ✓ Started: {fname}.csv")
    except Exception as e:
        print(f"  ✗ FAILED {hsa_id}: {e}")

print(f"\n{len(all_tasks)} tasks submitted")


Exporting 19 HSAs × 610 days → Drive/INF_DAILY_CLIMATE_V8

  ✓ Started: INF_HSA_Al-Basheer_Hospital_daily.csv
  ✓ Started: INF_HSA_Princess_Raya_Hospital_daily.csv
  ✓ Started: INF_HSA_AL-Shuneh_Hospital_daily.csv
  ✓ Started: INF_HSA_AL-Nadeem_Hospital_daily.csv
  ✓ Started: INF_HSA_AL-Zarqa_Hospital_daily.csv
  ✓ Started: INF_HSA_Khazzan_Primary_Center_daily.csv
  ✓ Started: INF_HSA_Jarash_Hospital_daily.csv
  ✓ Started: INF_HSA_Al-Yarmouk_Hospital_daily.csv
  ✓ Started: INF_HSA_Dr__Jamel_Al-Totanji_Hospital_daily.csv
  ✓ Started: INF_HSA_AL-Ramtha_Hospital_daily.csv
  ✓ Started: INF_HSA_Al-Mafraq_Gynecology_and_Pediatrics_Hospital_daily.csv
  ✓ Started: INF_HSA_Al-Karak_Hospital_daily.csv
  ✓ Started: INF_HSA_Mabroukeh_Primary_Center__daily.csv
  ✓ Started: INF_HSA_Tafilah_Governmental_Hospital_daily.csv
  ✓ Started: INF_HSA_Al_Hussain_New_Salt_Hospital_daily.csv
  ✓ Started: INF_HSA_Prince_Hashem_Primary_Center__daily.csv
  ✓ Started: INF_HSA_Jadaa_Primary_Center_daily.csv
  ✓ Star

In [ ]:
# STEP 6 — Wait for exports, then download
import os, time, re as _re
from datetime import datetime
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

LOCAL_DIR = os.path.join(OUT_DIR, f"DRIVE_CLIMATE_BY_HSA_DOWNLOAD_DAILY_{BOUNDARY_VERSION.upper()}")
os.makedirs(LOCAL_DIR, exist_ok=True)

CLIENT_SECRETS = "client_secrets.json"
SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]

flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRETS, SCOPES)
creds = flow.run_local_server(port=0)
drive_svc = build("drive", "v3", credentials=creds)

def get_folder_id(svc, name):
    q = f"mimeType='application/vnd.google-apps.folder' and name='{name}' and trashed=false"
    res = svc.files().list(q=q, fields="files(id,name)").execute()
    return res["files"][0]["id"]

def list_folder(svc, folder_id):
    items, token = [], None
    while True:
        res = svc.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            pageToken=token,
            fields="nextPageToken,files(id,name)",
            pageSize=1000,
        ).execute()
        items.extend(res.get("files", []))
        token = res.get("nextPageToken")
        if not token:
            break
    return {f["name"]: f["id"] for f in items}

folder_id = get_folder_id(drive_svc, DRIVE_FOLDER)
expected  = set(all_expected)

# Poll until all tasks complete and files appear on Drive.
# If all tasks reach COMPLETED but files are still missing after max_stall_polls
# consecutive polls, report fuzzy Drive matches and raise.
MAX_STALL_POLLS = 10
stall_polls = 0
drive_files = {}

while True:
    states = [t.status().get("state") for t in all_tasks]
    done   = sum(s == "COMPLETED" for s in states)
    failed = [s for s in states if s in {"FAILED", "CANCELLED"}]
    drive_files = list_folder(drive_svc, folder_id)
    have = len(set(drive_files) & expected)
    missing = sorted(expected - set(drive_files))
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{now}] Tasks {done}/{len(all_tasks)} complete | Drive files {have}/{len(expected)}")
    if failed:
        raise RuntimeError(f"Export(s) failed/cancelled: {failed}")
    if not missing:
        print(f"[{now}] ✓ All {len(expected)} files present.")
        break
    if done == len(all_tasks):
        stall_polls += 1
        print(f"  All tasks COMPLETED; {len(missing)} file(s) still missing "
              f"(stall poll {stall_polls}/{MAX_STALL_POLLS}).")
        if stall_polls >= MAX_STALL_POLLS:
            fuzzy = {}
            for mf in sorted(missing):
                stem = mf.rsplit(".", 1)[0] if "." in mf else mf
                ext  = "." + mf.rsplit(".", 1)[1] if "." in mf else ""
                cands = sorted(n for n in drive_files if n.startswith(stem) and n.endswith(ext))
                if cands:
                    fuzzy[mf] = cands
            print(f"\n[{now}] ⚠  Stalled: all tasks COMPLETED, {len(missing)} file(s) still absent.")
            for mf in sorted(missing):
                cands = fuzzy.get(mf, [])
                print(f"  {mf!r}  ->  {'possible match: ' + str(cands) if cands else 'no fuzzy match'}")
            raise RuntimeError(
                f"Stalled: {len(missing)} file(s) missing after {stall_polls} extra polls.\n"
                f"Missing: {missing[:10]}\n"
                "Likely cause: GEE Drive deduplication (' (N)' suffix). "
                "Rename the Drive file to the expected name and re-run the download block."
            )
    else:
        stall_polls = 0
    time.sleep(60)

# Download — with fuzzy fallback for deduplication suffixes
print("\nDownloading...")
for fname in sorted(expected):
    if fname in drive_files:
        fid = drive_files[fname]
    else:
        # Fuzzy match (e.g. ' (1)' suffix from Drive deduplication)
        stem = fname.rsplit(".", 1)[0] if "." in fname else fname
        ext  = "." + fname.rsplit(".", 1)[1] if "." in fname else ""
        cands = sorted(n for n in drive_files if n.startswith(stem) and n.endswith(ext))
        if not cands:
            raise FileNotFoundError(f"Cannot find Drive file for {fname!r}")
        fid = drive_files[cands[-1]]
        print(f"  Fuzzy: {fname!r} -> {cands[-1]!r}")
    path = os.path.join(LOCAL_DIR, fname)
    with open(path, "wb") as fh:
        dl = MediaIoBaseDownload(fh, drive_svc.files().get_media(fileId=fid))
        done_dl = False
        while not done_dl:
            _, done_dl = dl.next_chunk()
    print(f"  ✓ {fname}")

print(f"\nAll files downloaded to: {os.path.abspath(LOCAL_DIR)}")


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=26269363103-rostqdld5317cp32dmm4rjie4vvckd0s.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A63370%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly&state=TzAz68TtWceWASZ0kREjhi2DKKiwzL&access_type=offline
